# Symposia Getting Started

This notebook does one thing: run a live Symposia review.

Before you run it, set at least one of these environment variables:
- `OPENAI_API_KEY`
- `ANTHROPIC_API_KEY`
- `GOOGLE_API_KEY`

Run the cells from top to bottom.

In [1]:
import os
import sys
import time
from pathlib import Path

# Jupyter already has an event loop. This keeps live mode working in notebooks.
try:
    import nest_asyncio
except ImportError:
    nest_asyncio = None
else:
    nest_asyncio.apply()

ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(ROOT))

from symposia import validate

CLAIM = (
    "Vitamin C deficiency causes scurvy. "
    "Scurvy is a serious disease that can lead to bleeding gums and poor wound healing. "
    "Vitamin C supplementation prevents scurvy."
)

In [2]:
def show_summary(title, result):
    rows = list(result.aggregated_by_subclaim.values())
    escalated = [
        row.subclaim_id
        for row in rows
        if (
            row.support_score < 0.70
            or row.contradiction_score >= 0.35
            or row.sufficiency_score < 0.70
        )
    ]

    summary = {
        "run_id": result.run_id,
        "is_decisive": result.completion.is_decisive,
        "completion_reason": result.completion.reason,
        "aggregated_units": len(rows),
        "escalated_subclaims": escalated,
    }

    print(f"\n{title}")
    print("-" * len(title))
    for key, value in summary.items():
        print(f"{key:<20}: {value}")

## Run Live Review

This cell calls the configured provider and prints a short summary of the result.

In [3]:
providers = []
if os.getenv("OPENAI_API_KEY"):
    providers.append("openai")
if os.getenv("ANTHROPIC_API_KEY"):
    providers.append("anthropic")
if os.getenv("GOOGLE_API_KEY"):
    providers.append("google")

if not providers:
    print("No provider keys found.")
    print("Set OPENAI_API_KEY, ANTHROPIC_API_KEY, or GOOGLE_API_KEY, then rerun this cell.")
else:
    print(f"Using available provider key(s): {providers}")

    t0 = time.perf_counter()
    live_result = validate(
        content=f"{CLAIM} [nonce={time.time_ns()}]",
        domain="general",
        live=True,
    )
    elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)

    votes = live_result.core_trace.juror_votes
    provider_models = sorted({vote.provider_model for vote in votes if vote.provider_model})

    show_summary("Live Result", live_result)
    print(f"provider_models      : {provider_models if provider_models else 'unknown'}")
    print(f"execution_mode       : {live_result.execution_policy.get('juror_mode')}")
    print(f"elapsed_ms           : {elapsed_ms}")

Using available provider key(s): ['openai', 'anthropic', 'google']

Live Result
-----------
run_id              : run_faf756540f0c
is_decisive         : True
completion_reason   : initial_decisive
aggregated_units    : 1
escalated_subclaims : []
provider_models      : ['gpt-4o-mini']
execution_mode       : llm
elapsed_ms           : 1386.2


In [4]:
print("\nMore Detail")
print("-----------")
print(f"subclaim_ids         : {list(live_result.aggregated_by_subclaim.keys())}")
print(f"aggregation_rows     : {len(live_result.core_trace.aggregation_outcome)}")
print(f"runtime_stats        : {live_result.runtime_stats}")


More Detail
-----------
subclaim_ids         : ['sc_001']
aggregation_rows     : 1
runtime_stats        : {'juror_count': 1, 'total_decisions': 1, 'total_dropouts': 0, 'dropout_rate': 0.0}
